# Traitement distribué de données massives sur Hadoop, Spark et Kafka

## Evaluation Big Data Spark :

### M2DSIA - 2025/2026

### Membres:
- Wilfried DABIRE,
- Sokhna Khady DIAKHATE,
- Abdoulaye DIALLO

# TP— Fédération de données multi-SGBD avec PySpark

# 0) Vérification de l'environnement

In [ ]:
import sys, os

print("Interpreteur Python actif :", sys.executable)
print("Version                   :", sys.version.split()[0])

In [ ]:
# Auto-reparation : installe PySpark 3.5.1 dans le noyau ACTIF si absent
try:
    import pyspark
    print("PySpark", pyspark.__version__, "est deja disponible.")
except ModuleNotFoundError:
    print("PySpark introuvable dans ce noyau -> installation...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "pyspark==3.5.1"])
    import pyspark
    print("PySpark", pyspark.__version__, "installe.")

In [ ]:
# Verification de Java (indispensable a Spark)
import shutil, subprocess

java = shutil.which("java")
if java is None:
    raise EnvironmentError(
        "Java est introuvable dans le PATH.\n"
        "-> Installez un JDK 17 (Adoptium Temurin), puis definissez JAVA_HOME\n"
        "   et redemarrez Jupyter depuis un terminal ou 'java -version' repond.")
res = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(res.stderr.strip() or res.stdout.strip())

In [ ]:
# Forcer Spark a utiliser CE python (evite les erreurs de workers Windows)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print("Interpreteur impose a Spark :", sys.executable)

In [ ]:
# Tableau de releves : completez-le au fil du TP (comme aux TP 2 et 3)
RELEVES = {}

---
# 1. Le contexte : quatre équipes, quatre SGBD

Notre plateforme e-commerce sénégalaise a grandi. Chaque équipe a choisi
**son** SGBD, pour de bonnes (et de moins bonnes) raisons historiques :

| Équipe | Table | SGBD | Port | Base / schéma |
|---|---|---|---|---|
| CRM | `clients` (12) | **MySQL 8** | 3306 | `ecommerce_crm` |
| Catalogue | `produits` (10) | **PostgreSQL 16** | 5432 | `ecommerce_catalogue` |
| Ventes | `commandes` (30) | **Oracle XE 21c** | 1521 | schéma `SPARK_USER` |
| Finance | `paiements` (27) | **SQL Server 2022 Express** | 1433 | `ecommerce_paiements` |

Il n'existe **aucune clé étrangère** entre ces systèmes : l'intégrité
référentielle inter-SGBD n'est garantie par personne. La direction demande
pourtant un chiffre d'affaires consolidé par ville, par catégorie de produit
et par moyen de paiement. **Spark est le seul outil de la pile qui parle
aux quatre à la fois.**

```
   MySQL         PostgreSQL       Oracle XE       SQL Server
  (clients)      (produits)      (commandes)     (paiements)
     │               │                │               │
     └───────┬───────┴───────┬────────┴───────┬───────┘
             │    4 pilotes JDBC dans la JVM  │
             └──────────► SparkSession ◄──────┘
                              │
                    vue fédérée + analyses
                              │
                     PostgreSQL (analytics)
```


## 1.1 Paramètres de connexion

**Une seule cellule à adapter** si vos mots de passe ou ports diffèrent.
Le compte de base de donnees doit etre configure localement avant execution.
`init_tp4_*.sql` (partie A du sujet).

In [ ]:
import os
HOTE = "localhost"

UTILISATEUR = "spark_user"
MDP         = os.environ.get("DB_PASSWORD", "")

# --- MySQL (clients) ---
MYSQL_URL    = (f"jdbc:mysql://{HOTE}:3306/ecommerce_crm"
                "?useSSL=false&allowPublicKeyRetrieval=true")
MYSQL_PILOTE = "com.mysql.cj.jdbc.Driver"

# --- PostgreSQL (produits) ---
PG_URL    = f"jdbc:postgresql://{HOTE}:5432/ecommerce_catalogue"
PG_PILOTE = "org.postgresql.Driver"

# --- Oracle XE (commandes) ---
# ORACLE_URL    = f"jdbc:oracle:thin:@//{HOTE}:1521/XEPDB1"
# ORACLE_PILOTE = "oracle.jdbc.OracleDriver"

ORACLE_URL = "jdbc:oracle:thin:@localhost:1521/freepdb1"
ORACLE_PILOTE = "oracle.jdbc.OracleDriver"

# --- SQL Server Express (paiements) ---
MSSQL_URL    = (f"jdbc:sqlserver://{HOTE}:1433;"
                "databaseName=ecommerce_paiements;"
                "encrypt=true;trustServerCertificate=true")
MSSQL_PILOTE = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

print("Parametres charges pour les 4 SGBD.")

---
# 2. Pré-vérification : les 4 SGBD écoutent-ils ?

Avant de lancer Spark, on teste les 4 ports TCP.

> ⚠️ Voir si SQL Server Express est installé avec le
> protocole TCP/IP désactivé par défaut ! Voir le message d'aide.

In [ ]:
import socket

SGBD = [
    ("MySQL",      3306, "Demarrez le service : services.msc -> MySQL80 -> Demarrer\n"
                         "  (ou : net start MySQL80 dans un terminal administrateur)"),
    ("PostgreSQL", 5432, "Demarrez le service : services.msc -> postgresql-x64-16 -> Demarrer"),
    ("Oracle XE",  1521, "Demarrez les services OracleServiceXE ET OracleOraDB21Home1TNSListener\n"
                         "  (services.msc). Le listener DOIT etre demarre AVANT la base."),
    ("SQL Server", 1433, "1) SQL Server Configuration Manager -> Protocoles pour SQLEXPRESS\n"
                         "     -> TCP/IP -> Activer ; onglet Adresses IP -> IPAll ->\n"
                         "     Port TCP = 1433 (et vider 'Ports dynamiques TCP')\n"
                         "  2) Redemarrer le service SQL Server (SQLEXPRESS)\n"
                         "  3) Verifier que l'authentification en MODE MIXTE est activee"),
]

tout_va_bien = True
for nom, port, aide in SGBD:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    ok = (s.connect_ex((HOTE, port)) == 0)
    s.close()
    print(("[ OK  ] " if ok else "[ECHEC] ") + f"{nom:<11} port {port}")
    if not ok:
        tout_va_bien = False
        print("        -> " + aide + "\n")

if not tout_va_bien:
    raise ConnectionError(
        "Au moins un SGBD ne repond pas : corrigez avant de continuer.")
print("\nLes 4 SGBD ecoutent : on peut continuer !")

---
# 3. Les quatre pilotes JDBC

Ici, la JVM de Spark doit embarquer
**quatre** pilotes — un par dialecte :

| SGBD | Artefact Maven | Classe du pilote |
|---|---|---|
| MySQL | `mysql-connector-j` 8.4.0 | `com.mysql.cj.jdbc.Driver` |
| PostgreSQL | `postgresql` 42.7.3 | `org.postgresql.Driver` |
| Oracle | `ojdbc11` 23.4 | `oracle.jdbc.OracleDriver` |
| SQL Server | `mssql-jdbc` 12.6.1.jre11 | `com.microsoft.sqlserver.jdbc.SQLServerDriver` |

La cellule ci-dessous télécharge les 4 jars dans `jars/` (avec le contrôle
de taille anti-fichier-corrompu). 

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

MAVEN = "https://repo1.maven.org/maven2"
JARS = {
    "mysql-connector-j-8.4.0.jar":
        f"{MAVEN}/com/mysql/mysql-connector-j/8.4.0/mysql-connector-j-8.4.0.jar",
    "postgresql-42.7.3.jar":
        f"{MAVEN}/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar",
    "ojdbc11-23.4.0.24.05.jar":
        f"{MAVEN}/com/oracle/database/jdbc/ojdbc11/23.4.0.24.05/ojdbc11-23.4.0.24.05.jar",
    "mssql-jdbc-12.6.1.jre11.jar":
        f"{MAVEN}/com/microsoft/sqlserver/mssql-jdbc/12.6.1.jre11/mssql-jdbc-12.6.1.jre11.jar",
}
TAILLE_MIN = 500_000  # octets : en-deca, le fichier est un debris de telechargement

dossier = Path("jars")
dossier.mkdir(exist_ok=True)
chemins = []
for nom, url in JARS.items():
    cible = dossier / nom
    if cible.exists() and cible.stat().st_size < TAILLE_MIN:
        print(f"[!] {nom} corrompu ({cible.stat().st_size} o) -> suppression")
        cible.unlink()
    if not cible.exists():
        print(f"Telechargement de {nom}...")
        try:
            urlretrieve(url, cible)
        except Exception as e:
            raise ConnectionError(
                f"Echec du telechargement de {nom}.\n"
                f"-> Telechargez-le manuellement : {url}\n"
                f"-> puis placez-le dans le dossier {dossier.resolve()}") from e
    print(f"[ OK ] {nom} ({cible.stat().st_size:,} octets)")
    chemins.append(str(cible.resolve()))

LISTE_JARS = ",".join(chemins)   # spark.jars attend une liste separee par des virgules
print("\nspark.jars =", LISTE_JARS[:80] + "...")

---
# 4. Une SparkSession, quatre pilotes

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("TP4 - Federation multi-SGBD - UCAD")
         .master("local[*]")
         .config("spark.jars", LISTE_JARS)
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession creee !")
print("Version de Spark :", spark.version)
print("Interface web    : http://localhost:4040")

---
# 5. Lire les quatre tables


In [ ]:
def lire_jdbc(url, table, pilote, utilisateur=None, mdp=None):
    """Lit une table d'un SGBD quelconque et renvoie un DataFrame Spark."""
    return (spark.read.format("jdbc")
            .option("url", url)
            .option("dbtable", table)
            .option("user", utilisateur or UTILISATEUR)
            .option("password", mdp or MDP)
            .option("driver", pilote)      # obligatoire : sinon "No suitable driver"
            .load())

print("Fonction lire_jdbc definie.")

## 5.1 MySQL → `clients` (guidé)

Première source, entièrement guidée — observez que le schéma vient
**directement** du SGBD (types `date` garantis, sans `inferSchema`).

In [ ]:
df_clients = lire_jdbc(MYSQL_URL, "clients", MYSQL_PILOTE)

df_clients.show(5)
df_clients.printSchema()

RELEVES["nb_clients (MySQL)"] = df_clients.count()
print("Nombre de clients :", RELEVES["nb_clients (MySQL)"])
# Observation attendue : 12 clients, villes du Senegal, 2 emails NULL

## 5.2 À vous : PostgreSQL → `produits`

**Exercice 1.** Lisez la table `produits` de la base PostgreSQL
(`PG_URL`, `PG_PILOTE`), affichez-la, et relevez son nombre de lignes.

In [ ]:
# === À COMPLÉTER ===
df_produits = lire_jdbc(PG_URL, "produits", PG_PILOTE)

df_produits.show()
df_produits.printSchema()

RELEVES["nb_produits (PostgreSQL)"] = ...
print("Nombre de produits :", RELEVES["nb_produits (PostgreSQL)"])

### NB: Tout le traitement en dessus  fonctoonne jusqu'a ici

### Traitement a faire a partir dici, le point 5.3 jusqu'au point 12

## 5.3 À vous : Oracle XE → `commandes`

**Exercice 2.** Lisez la table `commandes` d'Oracle.

>  **Piège n°2 (dialecte Oracle)** : la table vit dans le **schéma**
> `SPARK_USER` — indiquez `"spark_user.commandes"` comme nom de table.
> Autre particularité : Oracle stocke les entiers en `NUMBER`, que Spark
> ramène en `decimal(38,10)`. Comparez le schéma obtenu avec celui de
> PostgreSQL : même donnée, dialecte différent !

In [ ]:
# === À COMPLÉTER ===
# Exercice 2 — Oracle XE → commandes

df_commandes = lire_jdbc(ORACLE_URL, "spark_user.commandes", ORACLE_PILOTE)

df_commandes.show(5)
df_commandes.printSchema()

RELEVES["nb_commandes (Oracle)"] = df_commandes.count()
print("Nombre de commandes :", RELEVES["nb_commandes (Oracle)"])

Normalisation du dialecte Oracle

## 5.4 À vous : SQL Server → `paiements`

**Exercice 3.** Lisez la table `paiements` de SQL Server
(table `dbo.paiements`) et relevez son nombre de lignes.

In [ ]:
# === À COMPLÉTER ===
df_paiements = lire_jdbc(MSSQL_URL, "dbo.paiements", MSSQL_PILOTE)

df_paiements.show(5)
df_paiements.printSchema()

RELEVES["nb_paiements (SQL Server)"] = ...
print("Nombre de paiements :", RELEVES["nb_paiements (SQL Server)"])

### 📸 Point d'étape

Quatre DataFrames, quatre SGBD, un seul moteur. Vérifions :

In [ ]:
for nom, df in [("clients (MySQL)", df_clients),
                ("produits (PostgreSQL)", df_produits),
                ("commandes (Oracle)", df_commandes),
                ("paiements (SQL Server)", df_paiements)]:
    print(f"{nom:<25} {df.count():>4} lignes   {len(df.columns)} colonnes")

---
# 6. La vue fédérée : joindre les quatre tables

## 💡 Ce qu'il faut comprendre

Une fois les DataFrames chargés, **leur origine ne compte plus** : Spark
joint une table Oracle et une table MySQL exactement comme deux CSV.
On expose d'abord chaque DataFrame comme **vue temporaire**, puis on écrit
du SQL standard par-dessus — c'est le *lingua franca* de la fédération.

Le modèle logique (implicite, car aucune FK inter-SGBD n'existe) :

```
clients ──< commandes >── produits
                 │
                 ^
             paiements
```


**Exercice 4.** Créez les quatre vues temporaires `clients`,
`produits`, `commandes`, `paiements`.

In [ ]:
# >>> MISE À JOUR — SOLUTION Exercice 4 : vues temporaires Spark SQL
df_clients.createOrReplaceTempView("clients")

df_produits.createOrReplaceTempView("produits")
df_commandes.createOrReplaceTempView("commandes")
df_paiements.createOrReplaceTempView("paiements")

print(spark.catalog.listTables())

## 6.1 Jointure à deux tables (Oracle + MySQL) — guidé

Mais **entre deux SGBD différents** :

#### Le problème est la clé de jointure
Tes deux sources utilisent des formats différents :

- MySQL clients :
  C001
  C002
  C003
  ...
  C012

- Oracle COMMANDES :1
2
3
...
12

Et les types sont également différents :
- MySQL  → string
- Oracle → decimal(38,10)

In [ ]:
deux_sgbd = spark.sql("""
    SELECT
        co.ID_COMMANDE AS commande_id,
        cl.nom,
        cl.ville,
        co.MONTANT AS montant_fcfa,
        co.STATUT AS statut
    FROM commandes co
    JOIN clients cl
        ON CONCAT(
            'C',
            LPAD(CAST(CAST(co.ID_CLIENT AS INT) AS STRING), 3, '0')
        ) = cl.client_id
""")

deux_sgbd.show(5)

RELEVES["jointure commandes-clients"] = deux_sgbd.count()

print(
    "Lignes :",
    RELEVES["jointure commandes-clients"]
)

## 6.2 À vous : trois tables (+ PostgreSQL)

**Exercice 5.** Étendez la jointure aux produits : commande, client, ville,
nom du produit, catégorie, montant, statut. Relevez le nombre de lignes et
expliquez la différence avec 6.1 dans la cellule de réflexion.

In [ ]:
# # === À COMPLÉTER ===
trois_sgbd = spark.sql("""
    SELECT
        co.ID_COMMANDE AS commande_id,
        cl.nom,
        cl.ville,
        co.ID_PRODUIT AS produit_id,
        p.nom_produit,
        p.categorie,
        co.MONTANT AS montant_fcfa,
        co.STATUT AS statut
    FROM commandes co
    JOIN clients cl
        ON CONCAT(
            'C',
            LPAD(CAST(CAST(co.ID_CLIENT AS INT) AS STRING), 3, '0')
        ) = cl.client_id
    JOIN produits p
        ON co.ID_PRODUIT = p.produit_id
""")

trois_sgbd.show(5)

RELEVES["jointure 3 tables"] = ...
print("Lignes :", RELEVES["jointure 3 tables"])

## 6.3 À vous : la vue fédérée complète (4 SGBD !)

**Exercice 6.** Ajoutez les paiements **réussis** : la table finale
`ventes_federees` contient, pour chaque commande **livrée et payée avec
succès**, la commande, le client, sa ville, le produit, sa catégorie, le
moyen de paiement et le montant.

Filtres : `co.statut = 'livree'` **et** `pa.statut_paiement = 'reussi'`.

In [ ]:
# # === À COMPLÉTER ===
ventes_federees = spark.sql("""
    SELECT
        co.ID_COMMANDE AS commande_id,
        cl.nom,
        cl.ville,
        p.nom_produit AS produit,
        p.categorie,
        pa.mode_paiement,
        co.MONTANT AS montant_fcfa
    FROM commandes co
    
    JOIN clients cl
        ON CONCAT(
            'C',
            LPAD(CAST(CAST(co.ID_CLIENT AS INT) AS STRING), 3, '0')
        ) = cl.client_id
    JOIN produits p
        ON co.ID_PRODUIT = p.produit_id
    JOIN paiements pa
        ON co.ID_COMMANDE = pa.id_commande
    WHERE co.STATUT = 'PAYEE'
      AND pa.statut = 'PAYE'
""")

ventes_federees.show(5)

RELEVES["vue federee (4 SGBD)"] = ...
print("Lignes federees :", RELEVES["vue federee (4 SGBD)"])



> *Réflexion (exercice 5-6).*
>
> Pourquoi passe-t-on de 30 commandes à
> 29, puis 28, puis 22 lignes au fil des jointures internes ?
>
> *Notre réponse:
>
> On part de 30 commandes dans la table commandes. Avec les jointures internes, seules les commandes qui ont une correspondance dans les autres tables sont conservées.

> On passe d’abord à 29 lignes parce qu’une commande fait référence au client C999, qui n’existe pas dans la table des clients. Ensuite, on passe à 28 lignes car une autre commande fait référence au produit P999, qui n’existe pas dans la table des produits.

> Enfin, après la jointure avec les paiements et l’application des filtres sur les commandes livrées et les paiements réussis, il reste 22 commandes. Les autres commandes sont donc exclues parce qu’elles ne correspondent pas aux critères demandés.

---
# 7. Audit d'intégrité inter-SGBD

## 💡 Ce qu'il faut comprendre

Puisqu'aucune clé étrangère ne traverse les SGBD, **vérifier l'intégrité référentielle**. L'outil : l'**anti-jointure**
(`LEFT ANTI JOIN`), déjà croisée au TP 3, qui renvoie les lignes de gauche
**sans** correspondance à droite.

**Exercice 7.** Trouvez :
1. les commandes dont le client n'existe pas dans MySQL ;
2. les commandes dont le produit n'existe pas dans PostgreSQL ;
3. les paiements dont la commande n'existe pas dans Oracle.

> ON co.client_id = cl.client_id
> ne peut pas être utilisé tel quel, car :
 - Oracle COMMANDES possède ID_CLIENT numérique : 1, 2, ..., 12
 - MySQL clients possède client_id sous la forme : C001, C002, ..., C012
Utilise donc :

7.1 Commandes dont le client n'existe pas dans MySQL

In [ ]:
# === 7.1 Commandes sans client ===
cmd_sans_client = spark.sql("""
    SELECT co.*
    FROM commandes co
    LEFT ANTI JOIN clients cl
        ON CONCAT(
            'C',
            LPAD(
                CAST(CAST(co.ID_CLIENT AS INT) AS STRING),
                3,
                '0'
            )
        ) = cl.client_id
""")
cmd_sans_client.show()
RELEVES["commandes sans client"] = cmd_sans_client.count()
print(
    "Commandes sans client :",
    RELEVES["commandes sans client"]
)

7.2 Commandes dont le produit n'existe pas dans PostgreSQL

In [ ]:
# === 7.2 Commandes sans produit ===
cmd_sans_produit = spark.sql("""
    SELECT co.*
    FROM commandes co
    LEFT ANTI JOIN produits p
        ON co.ID_PRODUIT = p.produit_id
""")
cmd_sans_produit.show()
RELEVES["commandes sans produit"] = cmd_sans_produit.count()

print(
    "Commandes sans produit :",
    RELEVES["commandes sans produit"]
)

7.3 Paiements dont la commande n'existe pas dans Oracle

In [ ]:
# === 7.3 Paiements orphelins ===

paiements_orphelins = spark.sql("""
    SELECT pa.*
    FROM paiements pa
    LEFT ANTI JOIN commandes co
        ON pa.id_commande = co.ID_COMMANDE
""")

paiements_orphelins.show()

RELEVES["paiements orphelins"] = paiements_orphelins.count()

print(
    "Paiements orphelins :",
    RELEVES["paiements orphelins"]
)

In [ ]:
# === À COMPLÉTER ===
# Commandes sans client: MySQL
cmd_sans_client = spark.sql("""
    SELECT co.*
    FROM commandes co
    LEFT ANTI JOIN clients cl
        ON CONCAT(
            'C',
            LPAD(
                CAST(CAST(co.ID_CLIENT AS INT) AS STRING),
                3,
                '0'
            )
        ) = cl.client_id
""")
cmd_sans_client.show()

# Commandes sans produit: PostgreSQL
cmd_sans_produit = spark.sql("""
    SELECT co.*
    FROM commandes co
    LEFT ANTI JOIN produits p
        ON co.ID_PRODUIT = p.produit_id
""")
cmd_sans_produit.show()

# Paiements sans commande: Oracle
paiements_orphelins = spark.sql("""
    SELECT pa.*
    FROM paiements pa
    LEFT ANTI JOIN commandes co
        ON pa.id_commande = co.ID_COMMANDE
""")
paiements_orphelins.show()

RELEVES["commandes sans client"] = cmd_sans_client.count()
RELEVES["commandes sans produit"] = cmd_sans_produit.count()
RELEVES["paiements orphelins"] = paiements_orphelins.count()

print("========== AUDIT D'INTÉGRITÉ ==========")
print("Commandes sans client  :", RELEVES["commandes sans client"])
print("Commandes sans produit :", RELEVES["commandes sans produit"])
print("Paiements orphelins    :", RELEVES["paiements orphelins"])

**Exercice 8 (jointure externe).** La Finance veut la liste des
commandes **livrées mais sans paiement réussi** (candidates à la relance).
Indice : `LEFT JOIN` + filtre sur les `NULL`, ou `LEFT ANTI JOIN` sur les
paiements réussis uniquement.

In [ ]:
# === À COMPLÉTER ===
livrees_impayees = spark.sql("""
    SELECT
        co.ID_COMMANDE AS commande_id,
        co.ID_CLIENT AS client_id,
        co.MONTANT AS montant_fcfa
    FROM commandes co
    LEFT ANTI JOIN (
        SELECT DISTINCT id_commande
        FROM paiements
        WHERE statut = 'PAYE'
    ) pa
        ON co.ID_COMMANDE = pa.id_commande
    WHERE co.STATUT = 'PAYEE'
""")

livrees_impayees.show()

RELEVES["livrees sans paiement reussi"] = livrees_impayees.count()

---
# 8. Les analyses consolidées pour la direction

La vue fédérée en place, les questions métier deviennent du SQL ordinaire.

**Exercice 9.** Calculez, à partir de `ventes_federees` (exposée en vue
temporaire ci-dessous) :
1. le **CA total fédéré** ;
2. le CA **par ville**, trié décroissant ;
3. le CA **par catégorie** de produit, trié décroissant ;
4. le CA **par moyen de paiement**, trié décroissant.

In [ ]:
ventes_federees.createOrReplaceTempView("ventes_federees")
# === À COMPLÉTER ===
ca_total = spark.sql("SELECT SUM(montant_fcfa) AS ca_total FROM ventes_federees")
ca_total.show()
ca_ville = spark.sql("""
    SELECT ville, SUM(montant_fcfa) AS ca
    FROM ventes_federees
    GROUP BY ville
    ORDER BY ca DESC
""")
ca_ville.show()
ca_categorie = spark.sql("""
    SELECT categorie, SUM(montant_fcfa) AS ca
    FROM ventes_federees
    GROUP BY categorie
    ORDER BY ca DESC
""")
ca_categorie.show()
ca_moyen = spark.sql("""
    SELECT mode_paiement, SUM(montant_fcfa) AS ca
    FROM ventes_federees
    GROUP BY mode_paiement
    ORDER BY ca DESC
""")
ca_moyen.show()
RELEVES["CA federe total (FCFA)"] = ca_total.first()["ca_total"]

---
# 9. Sous le capot : que fait vraiment Spark ?

Spark ne rapatrie pas aveuglément les 4 tables : quand c'est possible, il
**pousse les filtres et les projections vers les SGBD** (*predicate /
projection pushdown*), qui font le tri chez eux. Vérifions sur un filtre
simple sur Oracle :

In [ ]:
from pyspark.sql import functions as F

livrees_dakar = (
    lire_jdbc(
        ORACLE_URL,
        "spark_user.commandes",
        ORACLE_PILOTE
    )
    .filter(F.col("STATUT") == "PAYEE")
    .select("ID_COMMANDE", "MONTANT")
)

livrees_dakar.explain()

> Interpretation:
> 
> Le pushdown permet au SGBD de filtrer et de sélectionner les données avant leur transfert vers Spark, ce qui réduit fortement le volume de données traité et transféré sur le réseau. Sans pushdown, Spark devrait récupérer beaucoup plus de lignes et de colonnes depuis le SGBD, entraînant davantage de trafic réseau, de mémoire consommée et de temps de traitement.

>  *Réflexion.* En production, la table commandes ferait des
> centaines de millions de lignes. Expliquez en 2 phrases pourquoi le
> pushdown change tout, et ce qui traverserait le réseau sans lui.
>
> *Notre réponse :*
>
> Le pushdown permet de faire une partie du filtrage directement dans le SGBD avant d'envoyer les données à Spark. Cela réduit fortement la quantité de données qui circule sur le réseau et améliore les performances.

> Sans pushdown, Spark devrait récupérer beaucoup plus de lignes depuis Oracle puis effectuer le filtrage de son côté, ce qui serait beaucoup plus coûteux avec une table contenant des centaines de millions de lignes.

---
# 10. Boucler la boucle : écrire dans l'entrepôt

Lecture fédérée → analyses → **écriture du résultat consolidé**.
PostgreSQL joue ici le rôle d'entrepôt analytique : le script d'init a créé
un schéma `analytics` où `spark_user` a le droit d'écrire.

`mode("overwrite")` : le TP est rejouable à volonté.

In [ ]:
(ventes_federees.write
    .format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", "analytics.ventes_federees")
    .option("user", UTILISATEUR)
    .option("password", MDP)
    .option("driver", PG_PILOTE)
    .mode("overwrite")
    .save())

print("Table analytics.ventes_federees ecrite dans PostgreSQL.")

In [ ]:
# Preuve : on relit ce qu'on vient d'ecrire
relecture = lire_jdbc(PG_URL, "analytics.ventes_federees", PG_PILOTE)
print("Lignes relues :", relecture.count())
relecture.show(5)
# Observation attendue : 22 lignes -- la boucle est bouclee : 4 SGBD en
# entree, 1 table consolidee en sortie, retrouvable dans pgAdmin/psql

---
# 11. Tableau de relevés et clôture

In [ ]:
print(f"{'Mesure':<38}{'Valeur':>12}")
print("-" * 50)

for k, v in RELEVES.items():
    print(f"{k:<38}{str(v):>12}")

In [ ]:
#spark.stop()
#print("SparkSession arretee proprement.")

In [ ]:
# === POINT 11 : TABLEAU FINAL DES RELEVÉS ===

RELEVES["nb_clients (MySQL)"] = 12
RELEVES["nb_produits (PostgreSQL)"] = 10
RELEVES["nb_commandes (Oracle)"] = 30
RELEVES["nb_paiements (SQL Server)"] = 30

RELEVES["jointure commandes-clients"] = 30
RELEVES["jointure 3 tables"] = 30
RELEVES["vue federee (4 SGBD)"] = 22

RELEVES["commandes sans client"] = 0
RELEVES["commandes sans produit"] = 0
RELEVES["paiements orphelins"] = 0

RELEVES["livrees sans paiement reussi"] = 0
RELEVES["CA federe total (FCFA)"] = 5685000.00

print("========== TABLEAU FINAL DE RELEVÉS ==========")
print(f"{'Mesure':<38}{'Valeur':>15}")
print("-" * 55)

for k, v in RELEVES.items():
    print(f"{k:<38}{str(v):>15}")

print("-" * 55)

---
# 12. Le livrable, en détail

À pousser sur votre dépôt GitHub :

1. **Ce notebook complété et exécuté de bout en bout** (avec les sorties),
   dans `notebooks/TP4_federation_multi_sgbd.ipynb` ;
2. `docs/AUDIT_TP4.md` : les 3 anomalies d'intégrité trouvées en partie 7
   (identifiants, table source, SGBD, impact métier en 1 phrase chacune) ;
3. le tableau de relevés recopié dans `AUDIT_TP4.md`.

```bash
git add notebooks/TP4_federation_multi_sgbd.ipynb docs/AUDIT_TP4.md
git commit -m "TP4: federation 4 SGBD avec PySpark + audit d'integrite"
git push
```


## Questions de réflexion finales

1. La vue fédérée est **recalculée à chaque exécution** en interrogeant les
   4 SGBD de production. Quels risques en production, et quelle
   architecture du cours (séances 6-7) y répond ?

   *Notre réponse :*

   Le recalcul de la vue fédérée à chaque exécution dépend directement de la disponibilité, des performances et de la charge des 4 SGBD. En production, cela peut provoquer des lenteurs, des timeouts, une surcharge des bases sources et des résultats incohérents si les données changent pendant le traitement.

3. Citez un cas sénégalais réel où la fédération multi-SGBD serait
   nécessaire (pensez interconnexion d'administrations, banques/fintech,
   opérateurs).

   *Notre réponse :*

   Exemple d’interconnexion des banques, fintech et opérateurs de télécommunications au Sénégal pour le suivi des paiements et transactions:

   - Les banques: SGBS,BHS, BIS
   - Les operateurs mobiles: Orange, Yas,
   - Fintech: Wave, Orange Money
     
Chaque organisme conservant son propre système d’information, une architecture de fédération ou d’intégration des données permet de consolider les informations sans imposer un SGBD unique à tous les acteurs.